# 03 — Vanilla online SGD baseline

## Goal

Implement and verify the source-compatible **mechanism** for vanilla batch-size-one SGD:
consume one Phase 1 experience, predict, compute an explicit scalar loss, backpropagate,
update immediately, and move to the next experience.

This notebook performs a short engineering smoke run on the exact 40.97-million-parameter
architecture. It does not claim a final paper-comparable learning result because the
official source omits essential training hyperparameters.


## Source audit

The official Oak Lab note,
[Learning from experience instead of curated datasets](https://www.oaklab.ai/posts/learning-from-experience-instead-of-curated-datasets),
says that a linear demonstration follows the gradient of the “sample mean squared
prediction error,” and later states that NoisyMNIST weights are learned with SGD and
NetworkIDBD. It does not give an explicit NoisyMNIST loss equation or the experimental
hyperparameters needed for a faithful full run.

No official supplement or source implementation was found in the workspace or linked from
the note.


## Paper-defined decisions

- The stream emits one experience at each time step.
- SGD is used as a comparison optimizer.
- NoisyMNIST supplies the inputs and noisy prediction targets.
- The exact architecture and initialization are those verified in Phase 2.

## Reproduction-protocol decisions

- One immediate batch-size-one update per experience is the minimum-intervention online
  protocol required for this reproduction; the note does not explicitly state the NoisyMNIST
  optimizer update cadence.
- The only source-described objective phrase, sample mean squared prediction error, is
  represented provisionally as full scalar squared error: (prediction − target)².
- An explicit loss enum prevents confusing it with half-squared error.
- A learning rate of 0.001 and eight updates are used **only for an engineering smoke test**.
- Pre/post smoke diagnostics use 32 fixed validation experiences.
- SGD explicitly disables momentum, dampening, Nesterov, weight decay, maximize, and
  foreach updates.

These smoke values are not selected hyperparameters and are not attributed to the source.

## Open questions

- UNRESOLVED_FROM_SOURCE: whether NoisyMNIST uses squared error, half-squared error, or
  another exact scaling.
- UNRESOLVED_FROM_SOURCE: SGD learning rate or candidate grid.
- UNRESOLVED_FROM_SOURCE: number of online updates.
- UNRESOLVED_FROM_SOURCE: evaluation cadence and metric.
- UNRESOLVED_FROM_SOURCE: number of independent runs.

The official test partition remains embargoed. This notebook loads only the official MNIST
training partition and its Phase 1 train/validation split.


## Reusable implementation

Phase 3 uses:

- [src/data/noisymnist.py](src/data/noisymnist.py), an importable port of the frozen
  Phase 1 definitions with identical constants and stochastic draw order;
- [src/models/noisymnist_mlp.py](src/models/noisymnist_mlp.py), verified in Phase 2;
- [src/training/online_sgd.py](src/training/online_sgd.py), the transparent update and
  evaluation routines.

Notebook 01 is not executed or modified. The importable stream was independently compared
against its definitions for the first 128 canonical training experiences before this
notebook was finalized. All Phase 1 tensor and metadata fields were bit-identical.
`pool_name` is an added non-stochastic provenance field, and the golden hash below is a
fingerprint of this importable port rather than Notebook 01's original hash.


In [1]:
from __future__ import annotations

import gc
import hashlib
import math
import platform
import time
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from src.data import (
    ExperimentConfig,
    NoisyMNISTStream,
    experiences_equal,
    make_fixed_stream,
    make_train_validation_partitions,
    sequences_equal,
)
from src.models import ModelConfig, NoisyMNISTMLP
from src.training import (
    LossConvention,
    evaluate_model,
    make_vanilla_sgd,
    online_sgd_update,
    prediction_loss,
)
from src.utils import seed_everything

print(f"Python:  {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")


Python:  3.12.14
PyTorch: 2.13.0+cpu
CUDA:    False


## Configuration

The Phase 1 split and canonical stream seeds are preserved exactly. The Phase 2
initialization seed is also reused exactly. Smoke-only settings are separated visibly.


In [2]:
DATA_ROOT = Path("data")
NOTEBOOK_01_PATH = Path("01_noisymnist_dataset_and_protocol.ipynb")
EXPECTED_NOTEBOOK_01_SHA256 = (
    "0d93ccdf489c16b56e2ad90b6b4ac47913177055701d2d0a9e279f93062dc409"
)
EXPECTED_CANONICAL_128_STREAM_SHA256 = (
    "75f48e433016a526fe5a0b697daad11cc24d651349bfc85d4b10927c5a95621f"
)

MODEL_INITIALIZATION_SEED = 20_260_202

# REPRODUCTION_PROTOCOL: engineering smoke values, not paper hyperparameters.
SMOKE_LEARNING_RATE = 1e-3
SMOKE_TRAINING_STEPS = 8
SMOKE_VALIDATION_STEPS = 32
SMOKE_LOSS_CONVENTION = LossConvention.MEAN_SQUARED_ERROR

execution_device = "cuda" if torch.cuda.is_available() else "cpu"
data_config = ExperimentConfig(device=execution_device)
model_config = ModelConfig(
    initialization_seed=MODEL_INITIALIZATION_SEED,
    use_bias=False,
    bias_initial_value=None,
    dtype=torch.float32,
    device=execution_device,
)
seed_everything(MODEL_INITIALIZATION_SEED)

protocol_display = pd.Series(
    {
        "split_seed": data_config.split_seed,
        "train_stream_seed": data_config.train_stream_seed,
        "validation_stream_seed": data_config.validation_stream_seed,
        "model_initialization_seed": MODEL_INITIALIZATION_SEED,
        "smoke_learning_rate": SMOKE_LEARNING_RATE,
        "smoke_training_steps": SMOKE_TRAINING_STEPS,
        "smoke_validation_steps": SMOKE_VALIDATION_STEPS,
        "smoke_loss_convention": SMOKE_LOSS_CONVENTION.value,
        "device": execution_device,
        "dtype": str(model_config.dtype),
    },
    name="value",
)
display(protocol_display.to_frame())


,value
split_seed,20260201
train_stream_seed,314159
validation_stream_seed,271828
model_initialization_seed,20260202
smoke_learning_rate,0.001
smoke_training_steps,8
smoke_validation_steps,32
smoke_loss_convention,mean_squared_error
device,cpu
dtype,torch.float32


## Phase 1 compatibility and leakage checks

The frozen Notebook 01 hash protects against accidental modification. Golden per-class
split counts and a canonical 128-experience fingerprint protect the imported protocol.
The fingerprint includes every image/target tensor and every metadata field.


In [3]:
notebook_01_sha256 = hashlib.sha256(NOTEBOOK_01_PATH.read_bytes()).hexdigest()
notebook_01_unchanged_pass = notebook_01_sha256 == EXPECTED_NOTEBOOK_01_SHA256
assert notebook_01_unchanged_pass

partitions = make_train_validation_partitions(
    DATA_ROOT,
    config=data_config,
    download=False,
)
training_pool = partitions.training_pool
validation_pool = partitions.validation_pool

train_index_set = set(partitions.train_indices.tolist())
validation_index_set = set(partitions.validation_indices.tolist())
train_counts = np.bincount(
    partitions.dataset.targets[partitions.train_indices].cpu().numpy(),
    minlength=10,
)
validation_counts = np.bincount(
    partitions.dataset.targets[partitions.validation_indices].cpu().numpy(),
    minlength=10,
)
expected_train_counts = np.array(
    [5429, 6180, 5462, 5620, 5355, 4969, 5425, 5743, 5364, 5453]
)
expected_validation_counts = np.array(
    [494, 562, 496, 511, 487, 452, 493, 522, 487, 496]
)

partition_compatibility_pass = bool(
    partitions.dataset.train is True
    and len(training_pool) == 55_000
    and len(validation_pool) == 5_000
    and not train_index_set.intersection(validation_index_set)
    and train_index_set.union(validation_index_set) == set(range(60_000))
    and np.array_equal(train_counts, expected_train_counts)
    and np.array_equal(validation_counts, expected_validation_counts)
)
test_embargo_pass = bool(
    training_pool.source_partition == "official_train"
    and validation_pool.source_partition == "official_train"
    and training_pool.name == "training"
    and validation_pool.name == "validation"
    and "test_pool" not in globals()
)
assert partition_compatibility_pass
assert test_embargo_pass


def stream_fingerprint(stream: NoisyMNISTStream, num_steps: int) -> str:
    """Hash a finite ordered stream, including tensors and metadata."""

    digest = hashlib.sha256()
    tensor_keys = (
        "x",
        "x_flat",
        "noisy_target",
        "clean_target",
        "gaussian_noise",
    )
    metadata_keys = (
        "digit_present",
        "digit_label",
        "mnist_index",
        "source_partition",
        "pool_name",
        "stream_step",
    )
    for _ in range(num_steps):
        experience = next(stream)
        for key in tensor_keys:
            digest.update(
                experience[key]
                .detach()
                .cpu()
                .contiguous()
                .numpy()
                .tobytes()
            )
        for key in metadata_keys:
            digest.update(repr(experience[key]).encode("utf-8"))
            digest.update(bytes([0]))
    return digest.hexdigest()


fingerprint_stream = NoisyMNISTStream(
    training_pool,
    data_config,
    data_config.train_stream_seed,
)
canonical_stream_sha256 = stream_fingerprint(fingerprint_stream, 128)
canonical_stream_compatibility_pass = (
    canonical_stream_sha256 == EXPECTED_CANONICAL_128_STREAM_SHA256
)
assert canonical_stream_compatibility_pass

print(f"Frozen Notebook 01 SHA-256: {notebook_01_sha256}")
print(f"Canonical 128-stream SHA-256: {canonical_stream_sha256}")
print("Official test partition was not loaded.")


Frozen Notebook 01 SHA-256: 0d93ccdf489c16b56e2ad90b6b4ac47913177055701d2d0a9e279f93062dc409
Canonical 128-stream SHA-256: 75f48e433016a526fe5a0b697daad11cc24d651349bfc85d4b10927c5a95621f
Official test partition was not loaded.


## Paired-stream and checkpoint-state tests

Two independent streams with the same pool and seed must produce identical ordered
experiences. Stream-state restoration must also reproduce the exact future sequence.


In [4]:
paired_stream_a = NoisyMNISTStream(
    training_pool, data_config, data_config.train_stream_seed
)
paired_stream_b = NoisyMNISTStream(
    training_pool, data_config, data_config.train_stream_seed
)
paired_prefix_a = [next(paired_stream_a) for _ in range(32)]
paired_prefix_b = [next(paired_stream_b) for _ in range(32)]
paired_stream_identity_pass = sequences_equal(paired_prefix_a, paired_prefix_b)

paired_stream_a.reset()
paired_stream_b.reset()
paired_reset_pass = paired_stream_a.stream_step == paired_stream_b.stream_step == 0

state_probe = NoisyMNISTStream(
    training_pool, data_config, data_config.train_stream_seed
)
_ = [next(state_probe) for _ in range(11)]
saved_stream_state = state_probe.state_dict()
future_a = [next(state_probe) for _ in range(16)]
state_probe.load_state_dict(saved_stream_state)
future_b = [next(state_probe) for _ in range(16)]
stream_state_restore_pass = sequences_equal(future_a, future_b)

fixed_validation_a = list(
    make_fixed_stream(
        validation_pool,
        num_steps=16,
        seed=data_config.validation_stream_seed,
        config=data_config,
    )
)
fixed_validation_b = list(
    make_fixed_stream(
        validation_pool,
        num_steps=16,
        seed=data_config.validation_stream_seed,
        config=data_config,
    )
)
fixed_validation_identity_pass = sequences_equal(
    fixed_validation_a, fixed_validation_b
)

assert paired_stream_identity_pass
assert paired_reset_pass
assert stream_state_restore_pass
assert fixed_validation_identity_pass

del paired_prefix_a, paired_prefix_b, future_a, future_b
del fixed_validation_a, fixed_validation_b
gc.collect()
print("Paired streams, resets, state restoration, and fixed validation all passed.")


Paired streams, resets, state restoration, and fixed validation all passed.

## Exact objective convention

For a scalar output, “mean squared error” reduces to:

L_MSE = (prediction − target)²

Its derivative with respect to the prediction is 2(prediction − target). In contrast,
half-squared error has derivative prediction − target. This factor of two is scientifically
consequential because it rescales every effective SGD update.

The NoisyMNIST-specific convention is unresolved in the source. The smoke run uses full
squared error only because it matches the source's sole objective phrase; this is a
provisional reproduction assumption, not a confirmed paper equation.


In [5]:
mse_prediction = torch.tensor(2.0, requires_grad=True)
zero_target = torch.tensor(0.0)
mse_loss_probe = prediction_loss(
    mse_prediction,
    zero_target,
    LossConvention.MEAN_SQUARED_ERROR,
)
mse_loss_probe.backward()

half_prediction = torch.tensor(2.0, requires_grad=True)
half_loss_probe = prediction_loss(
    half_prediction,
    zero_target,
    LossConvention.HALF_SQUARED_ERROR,
)
half_loss_probe.backward()

loss_convention_test_pass = bool(
    float(mse_loss_probe.detach()) == 4.0
    and float(mse_prediction.grad) == 4.0
    and float(half_loss_probe.detach()) == 2.0
    and float(half_prediction.grad) == 2.0
)
assert loss_convention_test_pass

loss_convention_table = pd.DataFrame(
    {
        "loss at prediction=2,target=0": [
            float(mse_loss_probe.detach()),
            float(half_loss_probe.detach()),
        ],
        "dL/dprediction": [
            float(mse_prediction.grad),
            float(half_prediction.grad),
        ],
    },
    index=["mean squared error", "half-squared error"],
)
display(loss_convention_table)


,"loss at prediction=2,target=0",dL/dprediction
mean squared error,4.0,4.0
half-squared error,2.0,2.0


## Vanilla SGD construction

The optimizer is plain torch SGD with one learning rate and no auxiliary mechanisms. It
uses no momentum buffer and therefore has zero persistent tensor state.


In [6]:
smoke_model = NoisyMNISTMLP(model_config)
smoke_optimizer = make_vanilla_sgd(smoke_model, SMOKE_LEARNING_RATE)

optimizer_defaults = smoke_optimizer.defaults
vanilla_sgd_configuration_pass = bool(
    optimizer_defaults["lr"] == SMOKE_LEARNING_RATE
    and optimizer_defaults["momentum"] == 0.0
    and optimizer_defaults["dampening"] == 0.0
    and optimizer_defaults["weight_decay"] == 0.0
    and optimizer_defaults["nesterov"] is False
    and optimizer_defaults["maximize"] is False
    and optimizer_defaults["foreach"] is False
    and len(smoke_optimizer.state) == 0
)
assert vanilla_sgd_configuration_pass

display(pd.Series(optimizer_defaults, name="value").to_frame())
print("Persistent optimizer-state entries:", len(smoke_optimizer.state))


,value
lr,0.001
momentum,0.0
dampening,0.0
weight_decay,0.0
nesterov,False
maximize,False
foreach,False
differentiable,False
fused,None


Persistent optimizer-state entries: 0


## Fixed validation before the smoke run

Evaluation reconstructs a fresh validation stream, runs under inference mode, performs no
backward pass, and does not advance the training stream. A full-parameter hash verifies
that evaluation cannot silently modify the model.


In [7]:
def parameter_sha256(model: NoisyMNISTMLP, chunk_size: int = 1_000_000) -> str:
    """Hash all model parameter bytes with bounded temporary memory."""

    digest = hashlib.sha256()
    for parameter in model.parameters():
        flat_parameter = parameter.detach().reshape(-1)
        for start in range(0, flat_parameter.numel(), chunk_size):
            chunk = (
                flat_parameter[start : start + chunk_size]
                .cpu()
                .contiguous()
                .numpy()
            )
            digest.update(chunk.tobytes())
    return digest.hexdigest()


smoke_train_stream = NoisyMNISTStream(
    training_pool,
    data_config,
    data_config.train_stream_seed,
)
assert smoke_train_stream.pool.name == "training"
training_position_before_validation = smoke_train_stream.stream_step
parameter_hash_before_validation = parameter_sha256(smoke_model)

validation_before_stream = NoisyMNISTStream(
    validation_pool,
    data_config,
    data_config.validation_stream_seed,
)
validation_before = evaluate_model(
    smoke_model,
    validation_before_stream,
    num_steps=SMOKE_VALIDATION_STEPS,
    expected_pool_name="validation",
)

parameter_hash_after_validation = parameter_sha256(smoke_model)
evaluation_noninterference_pass = bool(
    parameter_hash_before_validation == parameter_hash_after_validation
    and smoke_train_stream.stream_step == training_position_before_validation
    and validation_before_stream.stream_step == SMOKE_VALIDATION_STEPS
    and validation_before.finite
)
assert evaluation_noninterference_pass
display(pd.Series(asdict(validation_before), name="before_smoke").to_frame())


,before_smoke
num_steps,32
noisy_target_mse,3.247481
clean_target_mse,0.09375
mean_prediction,0.0
finite,True


## Immediate batch-size-one updates

The first-layer gradient is exactly zero on the first update because all output weights
begin at zero. The first update changes the output layer; the second experience can then
propagate gradient into the first layer. The assertions below verify this chain-rule
behavior on the full faithful model.


In [8]:
first_layer_before_training = smoke_model.first_layer_weights.detach().clone()
second_layer_before_training = smoke_model.second_layer_weights.detach().clone()
smoke_step_metrics = []
smoke_step_seconds = []

for update_index in range(SMOKE_TRAINING_STEPS):
    experience = next(smoke_train_stream)
    assert smoke_train_stream.pool.name == "training"
    if experience["digit_present"]:
        assert experience["mnist_index"] in training_pool.index_set

    step_start = time.perf_counter()
    metrics = online_sgd_update(
        smoke_model,
        smoke_optimizer,
        experience,
        loss_convention=SMOKE_LOSS_CONVENTION,
        compute_gradient_norm=(update_index == 0),
        check_parameter_finiteness=(update_index == 0),
    )
    smoke_step_seconds.append(time.perf_counter() - step_start)
    smoke_step_metrics.append(metrics)

    if update_index == 0:
        first_update_first_layer_unchanged_pass = torch.equal(
            first_layer_before_training,
            smoke_model.first_layer_weights,
        )
        first_update_second_layer_changed_pass = not torch.equal(
            second_layer_before_training,
            smoke_model.second_layer_weights,
        )
    elif update_index == 1:
        second_update_first_layer_changed_pass = not torch.equal(
            first_layer_before_training,
            smoke_model.first_layer_weights,
        )
        del first_layer_before_training, second_layer_before_training
        gc.collect()

immediate_update_count_pass = (
    smoke_train_stream.stream_step == SMOKE_TRAINING_STEPS
    and len(smoke_step_metrics) == SMOKE_TRAINING_STEPS
)
momentum_free_state_pass = len(smoke_optimizer.state) == 0
trained_parameter_hash = parameter_sha256(smoke_model)
parameters_changed_pass = trained_parameter_hash != parameter_hash_before_validation
parameters_finite_pass = all(
    bool(torch.isfinite(torch.linalg.vector_norm(parameter.detach())))
    for parameter in smoke_model.parameters()
)

assert first_update_first_layer_unchanged_pass
assert first_update_second_layer_changed_pass
assert second_update_first_layer_changed_pass
assert immediate_update_count_pass
assert momentum_free_state_pass
assert parameters_changed_pass
assert parameters_finite_pass

smoke_metrics_frame = pd.DataFrame(
    [asdict(metrics) for metrics in smoke_step_metrics]
)
smoke_metrics_frame["step_seconds"] = smoke_step_seconds
display(smoke_metrics_frame)


,stream_step,prediction,noisy_target,clean_target,loss,noisy_squared_error,clean_squared_error,hidden_active_fraction,gradient_l2_norm,step_seconds
0,0,0.000000e+00,-1.060127,0.0,1.123870,1.123870,0.000000e+00,0.4971,0.049613,0.256634
1,1,-3.217182e-07,3.923115,0.0,15.390833,15.390834,1.035026e-13,0.4935,NaN,0.217148
2,2,6.641827e-07,-0.436901,0.0,0.190883,0.190883,4.411387e-13,0.4999,NaN,0.217595
3,3,7.275702e-07,-0.635124,0.0,0.403383,0.403383,5.293585e-13,0.5042,NaN,0.215081
4,4,4.581718e-07,-0.187352,0.0,0.035101,0.035101,2.099214e-13,0.5009,NaN,0.258133
5,5,3.786778e-07,-0.692999,0.0,0.480249,0.480249,1.433968e-13,0.4985,NaN,0.219544
6,6,1.096042e-07,1.500802,0.0,2.252406,2.252406,1.201309e-14,0.4901,NaN,0.211416
7,7,5.992626e-07,-0.569400,0.0,0.324217,0.324217,3.591157e-13,0.5043,NaN,0.216374


## Fixed validation after the smoke run

The same finite validation experiences are reconstructed from the same seed. These values
are diagnostics only; eight updates are far too few to estimate learning quality or select
a learning rate.


In [9]:
training_position_before_post_validation = smoke_train_stream.stream_step
parameter_hash_before_post_validation = parameter_sha256(smoke_model)
validation_after = evaluate_model(
    smoke_model,
    make_fixed_stream(
        validation_pool,
        num_steps=SMOKE_VALIDATION_STEPS,
        seed=data_config.validation_stream_seed,
        config=data_config,
    ),
    num_steps=SMOKE_VALIDATION_STEPS,
    expected_pool_name="validation",
)
parameter_hash_after_post_validation = parameter_sha256(smoke_model)

post_evaluation_noninterference_pass = bool(
    parameter_hash_before_post_validation == parameter_hash_after_post_validation
    and smoke_train_stream.stream_step == training_position_before_post_validation
    and validation_after.finite
)
assert post_evaluation_noninterference_pass

validation_comparison = pd.DataFrame(
    {
        "before_smoke": asdict(validation_before),
        "after_smoke": asdict(validation_after),
    }
)
display(validation_comparison)

steady_step_seconds = smoke_step_seconds[1:] or smoke_step_seconds
median_step_seconds = float(np.median(steady_step_seconds))
print(f"Median online-update time excluding first-step warm-up: {median_step_seconds:.4f} s")
print(
    "Mechanical projection for 10,000 CPU updates at this median: "
    f"{10_000 * median_step_seconds / 60:.1f} minutes"
)
print("This timing projection is not a chosen paper training horizon.")


,before_smoke,after_smoke
num_steps,32,32
noisy_target_mse,3.247481,3.247481
clean_target_mse,0.09375,0.09375
mean_prediction,0.0,0.0
finite,True,True


Median online-update time excluding first-step warm-up: 0.2171 s
Mechanical projection for 10,000 CPU updates at this median: 36.2 minutes
This timing projection is not a chosen paper training horizon.


## Verification checklist

All checklist values are derived from executed assertions.


In [10]:
phase3_checks = [
    ("Notebook 01 remained byte-for-byte unchanged", notebook_01_unchanged_pass),
    ("imported Phase 1 partitions match golden counts", partition_compatibility_pass),
    ("canonical stream matches golden fingerprint", canonical_stream_compatibility_pass),
    ("official test partition remained unloaded", test_embargo_pass),
    ("same-seed training streams are bit-identical", paired_stream_identity_pass),
    ("stream reset and checkpoint state are exact", paired_reset_pass and stream_state_restore_pass),
    ("fixed validation experiences are reproducible", fixed_validation_identity_pass),
    ("loss conventions have verified factor-of-two gradients", loss_convention_test_pass),
    ("optimizer is vanilla momentum-free SGD", vanilla_sgd_configuration_pass),
    ("pre-smoke validation did not alter model or training stream", evaluation_noninterference_pass),
    ("evaluation consumed exactly the requested experiences", validation_before_stream.stream_step == SMOKE_VALIDATION_STEPS),
    ("first update obeyed zero-output-layer chain rule", first_update_first_layer_unchanged_pass),
    ("second update reached the first layer", second_update_first_layer_changed_pass),
    ("one immediate update occurred per experience", immediate_update_count_pass),
    ("SGD persistent tensor state remained empty", momentum_free_state_pass),
    ("smoke parameters changed and remained finite", parameters_changed_pass and parameters_finite_pass),
    ("post-smoke validation did not alter model or training stream", post_evaluation_noninterference_pass),
]

for check_name, passed in phase3_checks:
    print(f"- [{'PASS' if passed else 'FAIL'}] {check_name}")
assert all(passed for _, passed in phase3_checks)


- [PASS] Notebook 01 remained byte-for-byte unchanged
- [PASS] imported Phase 1 partitions match golden counts
- [PASS] canonical stream matches golden fingerprint
- [PASS] official test partition remained unloaded
- [PASS] same-seed training streams are bit-identical
- [PASS] stream reset and checkpoint state are exact
- [PASS] fixed validation experiences are reproducible
- [PASS] loss conventions have verified factor-of-two gradients
- [PASS] optimizer is vanilla momentum-free SGD
- [PASS] pre-smoke validation did not alter model or training stream
- [PASS] evaluation consumed exactly the requested experiences
- [PASS] first update obeyed zero-output-layer chain rule
- [PASS] second update reached the first layer
- [PASS] one immediate update occurred per experience
- [PASS] SGD persistent tensor state remained empty
- [PASS] smoke parameters changed and remained finite
- [PASS] post-smoke validation did not alter model or training stream


## Results

The exact network successfully performed immediate batch-size-one SGD updates directly from
the canonical training stream. Fixed validation was deterministic, weight-neutral, and
independent of training-stream position. All optimizer, loss-scaling, chain-rule,
stream-identity, checkpoint-state, leakage, and finite-value tests passed.

The displayed pre/post metrics are only engineering diagnostics from eight updates. They
must not be interpreted as optimizer performance.

## Deviations from paper

- UNRESOLVED_FROM_SOURCE items prevent a final faithful SGD curve.
- Full squared error, learning rate 0.001, eight updates, and 32 validation samples are
  explicitly labeled smoke-test protocol choices.
- Biases remain disabled under the documented Phase 2 assumption.
- No test experiences were constructed or evaluated.

## Next phase

The online SGD implementation is mechanically ready. A defensible full baseline still
requires either official training details or a preregistered validation-only tuning
protocol for loss scaling, learning rate, horizon, cadence, and run count.

Phase 4 must derive NetworkIDBD from an authoritative source before any optimizer code is
written. The available Oak Lab note contains no NetworkIDBD equations, state definitions,
initialization, normalization, clipping, or meta-step-size, so implementation is currently
blocked pending that material.
